## MODELO PARA RESOLUCIÓN DE PROBLEMAS DE OPTIMIZACIÓN LINEAL CON REDES NEURONALES

Un problema de optimización lineal se plantea de la siguiente manera:

- Min $C^T y$   sujeto a $A y \geq b$

Donde:
- $C$ es el vector de costos
- $A$ es la matriz de restricciones
- $b$ es el vector de restricciones


Para enlazar esto con el problema de minimización del error en el aprendizaje automático de las Redes Neuronales, entonces se agregar el MSE al problema de optimización.

Se define la Nueva Función de Error como:


$$\mathcal{L} = \underbrace{||y - y_{target}||^2}_{\text{MSE}} + \lambda_1 \underbrace{||\text{ReLU}(Ay - b)||^2}_{\text{Constraint Penalty}} + \lambda_2 \underbrace{(\mathbf{c}^T y)}_{\text{LP Objective}}$$

Donde:
- $\lambda_1$ es el peso de la restricción
- $\lambda_2$ es el peso del objetivo



## Ejemplo

### The Scenario
You are building an app that suggests how much Steak and Beans a user should eat.

1. The Neural Network (MSE): Predicts what the user wants to eat based on their taste profile (e.g., the user loves Steak).

2. The LP Objective (Cost): We want the suggested meal to be cheap.

-   Steak costs $10.00 per unit.

-   Beans cost $1.00 per unit.

3. The Constraints (Health): The meal must have at least 20g of Protein.

-   Steak provides 10g protein/unit.

-   Beans provide 5g protein/unit.

### The Conflict
- User/MSE wants: Lots of Steak (Tasty).
- LP Objective wants: Only Beans (Cheap).
- Constraint wants: Enough food to hit 20g protein.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# --- 1. Setup Constants ---
# Costs (Objective Vector c)
# [Steak, Beans] -> Steak is $10, Beans is $1
cost_vector = torch.tensor([10.0, 1.0])

# Protein Content (Constraint Matrix A)
# [Steak, Beans] -> Steak=10g, Beans=5g
protein_content = torch.tensor([10.0, 5.0])

# Minimum Protein Required (b)
min_protein = 20.0

# User Preference (Target for MSE)
# The user LOVES Steak. They ideally want 3 units of Steak and 0 Beans.
user_preference = torch.tensor([[3.0, 0.0]])

In [ ]:
# --- 2. The Model ---
# A simple model that takes a "User ID" (dummy input) and predicts Food Amounts
model = nn.Sequential(
    nn.Linear(10, 10),
    nn.ReLU(),
    nn.Linear(10, 2),
    nn.ReLU() # Important: We can't eat negative food!
)
model

Sequential(
  (0): Linear(in_features=10, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=2, bias=True)
  (3): ReLU()
)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.02)

In [ ]:
# --- 3. The "Three-Head" Loss Function ---
def dietitian_loss(y_pred, y_target):
    # A. MSE Loss (Satisfaction)
    # Penalize deviating from what the user likes (3 Steak, 0 Beans)
    loss_satisfaction = nn.MSELoss()(y_pred, y_target)

    # B. LP Objective (Cost Minimization)
    # Calculate total bill: (Amount * Price)
    # We want this to be small.
    total_cost = torch.matmul(y_pred, cost_vector)
    loss_cost = torch.mean(total_cost)

    # C. LP Constraint (Health)
    # We need: Protein_Amount >= 20
    # Violation if: 20 - Protein_Amount > 0
    current_protein = torch.matmul(y_pred, protein_content)
    violation = torch.relu(min_protein - current_protein)
    loss_health = torch.mean(violation**2)

    # --- Weighted Sum ---
    # We weight "Health" highest because it's a hard rule.
    # We balance "Satisfaction" vs "Cost".
    return (1 * loss_satisfaction) + (0.5 * loss_cost) + (10.0 * loss_health)

In [ ]:
# --- 4. Training ---
dummy_input = torch.randn(1, 10)

print(f"{'Epoch':<5} | {'Steak':<8} | {'Beans':<8} | {'Protein':<8} | {'Cost ($)':<8}")
print("-" * 55)

for epoch in range(1000):
    optimizer.zero_grad()
    y_pred = model(dummy_input)

    loss = dietitian_loss(y_pred, user_preference)

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        d = y_pred[0].detach().numpy()
        prot = (d[0]*10 + d[1]*5)
        cost = (d[0]*10 + d[1]*1)
        print(f"{epoch:<5} | {d[0]:.2f}     | {d[1]:.2f}     | {prot:.2f}     | ${cost:.2f}")

# --- Final Analysis ---
print("-" * 55)
final = model(dummy_input)[0].detach()
print("FINAL RECOMMENDATION:")
print(f"Steak: {final[0]:.2f} units")
print(f"Beans: {final[1]:.2f} units")

Epoch | Steak    | Beans    | Protein  | Cost ($)
-------------------------------------------------------
0     | 0.00     | 0.06     | 0.29     | $0.06
20    | 2.10     | 3.27     | 37.36     | $24.29
40    | 2.89     | 4.23     | 50.10     | $33.18
60    | 2.94     | 4.25     | 50.60     | $33.62
80    | 2.84     | 4.08     | 48.81     | $32.48
100   | 2.72     | 3.89     | 46.61     | $31.06
120   | 2.59     | 3.69     | 44.33     | $29.59
140   | 2.46     | 3.49     | 42.07     | $28.12
160   | 2.34     | 3.30     | 39.87     | $26.68
180   | 2.22     | 3.11     | 37.74     | $25.29
200   | 2.10     | 2.94     | 35.70     | $23.95
220   | 1.99     | 2.77     | 33.77     | $22.67
240   | 1.88     | 2.62     | 31.93     | $21.45
260   | 1.78     | 2.47     | 30.19     | $20.30
280   | 1.69     | 2.34     | 28.55     | $19.20
300   | 1.60     | 2.21     | 27.00     | $18.17
320   | 1.51     | 2.09     | 25.53     | $17.18
340   | 1.43     | 1.97     | 24.15     | $16.26
360   | 1.35  